In [ ]:
import os

REPO_URL    = "https://github.com/NLightVN/haxball-agent-lite.git"
BRANCH      = "M-A-R-L"
REPO_DIR    = "/content/haxball-agent-lite"

if os.path.exists(REPO_DIR):
    print("Repo already exists — pulling latest...")
    !git -C {REPO_DIR} pull origin {BRANCH}
else:
    !git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

Git identity

In [ ]:
GIT_NAME  = "NLightVN"
GIT_EMAIL = "hung24378un@gmail.com"

!git config user.name  "{GIT_NAME}"
!git config user.email "{GIT_EMAIL}"
print("Git identity configured.")

requirement

In [ ]:
!pip install -q stable-baselines3[extra] sb3-contrib gymnasium numpy torch

Mount google drive for checkpoints

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Đường dẫn trên Drive ──────────────────────────────────────────────────────
DRIVE_BASE     = "/content/drive/MyDrive/haxball_A2v2_training"
DRIVE_A2V2_CKPT  = f"{DRIVE_BASE}/a2v2_checkpoints"

# Tạo thư mục trên Drive nếu chưa có
!mkdir -p {DRIVE_A2V2_CKPT}

# ── Symlink vào repo ──────────────────────────────────────────────────────────
LOCAL_MODELS = f"{REPO_DIR}/models"
!mkdir -p {LOCAL_MODELS}

# Xóa symlink/thư mục cũ (nếu có)
!rm -rf {LOCAL_MODELS}/a2v2_checkpoints

# Tạo symlink
!ln -s {DRIVE_A2V2_CKPT}  {LOCAL_MODELS}/a2v2_checkpoints

print("Symlinks created:")
!ls -la {LOCAL_MODELS}/

Training

In [ ]:
import os, glob

# ── Cấu hình ─────────────────────────────────────────────────────────────────
# Để resume sau disconnect: set CONTINUE_STEP = số step của checkpoint cuối cùng
# Ví dụ: CONTINUE_STEP = 500000  nếu có snapshot_500000.zip trên Drive
# Set = 0 để bắt đầu training mới (sẽ lấy file của A2v1 có sẵn trên repo làm gốc)
CONTINUE_STEP = 0

# ── Kiểm tra checkpoint A2v2 trên Drive ────────────────────────────────────────
ckpt_dir = f"{REPO_DIR}/models/a2v2_checkpoints"
ckpts = sorted(glob.glob(f"{ckpt_dir}/snapshot_*.zip"))
print(f"A2v2 checkpoints on Drive: {len(ckpts)} found")
for c in ckpts[-5:]:
    print(f"  {os.path.basename(c)}")

# ── Summary ───────────────────────────────────────────────────────────────────
print()
if CONTINUE_STEP > 0:
    resume_path = f"{ckpt_dir}/snapshot_{CONTINUE_STEP}.zip"
    exists = os.path.exists(resume_path)
    print(f"Mode: RESUME from step {CONTINUE_STEP:,}")
    print(f"  Checkpoint: {'✅' if exists else '❌ NOT FOUND!'} {os.path.basename(resume_path)}")
else:
    print("Mode: START FRESH (load A2v1 pretrained → fine-tune as A2v2)")

In [ ]:
import sys

train_script = f"{REPO_DIR}/training/train_a2v2.py"
base_arg = f"--base-model {REPO_DIR}/models/a2v2_checkpoints/snapshot_{CONTINUE_STEP}.zip" if CONTINUE_STEP > 0 else ""

print(f"Running: python {train_script} {base_arg}")
print("-" * 60)

!{sys.executable} {train_script} {base_arg}